In [1]:
!pip install requests beautifulsoup4 pandas fake-useragent tqdm -q
print("✓ Librerías instaladas")

✓ Librerías instaladas


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
import re
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ Importaciones correctas")
print(f"  Fecha de recolección: {datetime.now().strftime('%Y-%m-%d')}")

✓ Importaciones correctas
  Fecha de recolección: 2026-04-20


In [3]:
# Diccionario de destinos con sus URLs base en TripAdvisor
# Cada URL apunta a la página de reseñas del atractivo principal
DESTINOS = {
    'Galapagos': {
        'url_base': 'https://www.tripadvisor.com/Tourism-g294310-Galapagos_Islands_Province_of_Galapagos-Vacations.html',
        'categoria': 'Naturaleza',
        'region': 'Costa/Insular',
    },
    'Quito': {
        'url_base': 'https://www.tripadvisor.com/Attraction_Review-g294308-d313067-Reviews-Historic_Center_of_Quito-Quito_Pichincha_Province.html',
        'categoria': 'Ciudad/Patrimonio',
        'region': 'Sierra',
    },
    'Cuenca': {
        'url_base': 'https://www.tripadvisor.com/Tourism-g294309-Cuenca_Azuay_Province-Vacations.html',
        'categoria': 'Ciudad/Patrimonio',
        'region': 'Sierra',
    },
    'Banos': {
        'url_base': 'https://www.tripadvisor.com/Tourism-g635611-Banos_Tungurahua_Province-Vacations.html',
        'categoria': 'Aventura',
        'region': 'Sierra',
    },
    'Mindo': {
        'url_base': 'https://www.tripadvisor.com/Tourism-g2289626-Mindo_Pichincha_Province-Vacations.html',
        'categoria': 'Ecoturismo',
        'region': 'Sierra',
    },
    'Guayaquil': {
        'url_base': 'https://www.tripadvisor.com/Attraction_Review-g294307-d318616-Reviews-Malecon_2000-Guayaquil_Guayas_Province.html',
        'categoria': 'Ciudad/Costa',
        'region': 'Costa',
    },
    'Otavalo': {
        'url_base': 'https://www.tripadvisor.com/Attraction_Review-g635608-d318610-Reviews-Otavalo_Market-Otavalo_Imbabura_Province.html',
        'categoria': 'Cultura',
        'region': 'Sierra',
    },
    'Montanita': {
        'url_base': 'https://www.tripadvisor.com/Tourism-g635618-Montanita_Santa_Elena_Province-Vacations.html',
        'categoria': 'Playa',
        'region': 'Costa',
    },
}

print(f"✓ {len(DESTINOS)} destinos configurados:")
for nombre, info in DESTINOS.items():
    print(f"  - {nombre} ({info['categoria']}, {info['region']})")

✓ 8 destinos configurados:
  - Galapagos (Naturaleza, Costa/Insular)
  - Quito (Ciudad/Patrimonio, Sierra)
  - Cuenca (Ciudad/Patrimonio, Sierra)
  - Banos (Aventura, Sierra)
  - Mindo (Ecoturismo, Sierra)
  - Guayaquil (Ciudad/Costa, Costa)
  - Otavalo (Cultura, Sierra)
  - Montanita (Playa, Costa)


In [4]:
def get_headers():
    """Rota user agents para evitar bloqueos."""
    agents = [
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) '
        'Gecko/20100101 Firefox/121.0',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36',
    ]
    return {
        'User-Agent': random.choice(agents),
        'Accept-Language': 'en-US,en;q=0.9,es;q=0.8',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Referer': 'https://www.tripadvisor.com/',
    }


def extraer_resenas_pagina(soup, destino, categoria, region):
    """
    Extrae reseñas de una página de TripAdvisor parseada con BeautifulSoup.
    Retorna lista de diccionarios con los campos de cada reseña.
    """
    resenas = []

    # TripAdvisor cambia su estructura frecuentemente.
    # Intentamos múltiples selectores para mayor robustez.
    contenedores = (
        soup.find_all('div', {'data-automation': 'reviewCard'}) or
        soup.find_all('div', class_=re.compile(r'review-container|reviewSelector')) or
        soup.find_all('div', class_=re.compile(r'_c|YibKl'))
    )

    for item in contenedores:
        try:
            # Título
            titulo_tag = (
                item.find('span', {'data-automation': 'reviewTitle'}) or
                item.find('a', class_=re.compile(r'title|noQuotes'))
            )
            titulo = titulo_tag.get_text(strip=True) if titulo_tag else ''

            # Texto de la reseña
            texto_tag = (
                item.find('span', {'data-automation': 'reviewText'}) or
                item.find('p', class_=re.compile(r'partial_entry|reviewText')) or
                item.find('div', class_=re.compile(r'prw_rup|entry'))
            )
            texto = texto_tag.get_text(strip=True) if texto_tag else ''

            # Calificación (buscar en burbujas de rating)
            rating_tag = item.find(
                attrs={'aria-label': re.compile(r'\d+(\.\d+)? of 5 bubbles')})
            if rating_tag:
                rating_str = rating_tag.get('aria-label', '')
                rating_match = re.search(r'(\d+(?:\.\d+)?)', rating_str)
                rating = float(rating_match.group(1)) if rating_match else None
            else:
                # Alternativa: contar burbujas llenas
                burbujas = item.find_all(
                    'svg', class_=re.compile(r'bubble|UctUV'))
                rating = len(burbujas) if burbujas else None

            # Fecha
            fecha_tag = (
                item.find('span', {'data-automation': 'reviewedDate'}) or
                item.find('span', class_=re.compile(r'ratingDate|date'))
            )
            fecha_texto = fecha_tag.get_text(strip=True) if fecha_tag else ''

            # Solo guardar si tiene texto sustancial
            if texto and len(texto) > 20:
                resenas.append({
                    'destino':   destino,
                    'categoria': categoria,
                    'region':    region,
                    'titulo':    titulo,
                    'texto':     texto,
                    'rating':    rating,
                    'fecha_raw': fecha_texto,
                    'fecha_extraccion': datetime.now().strftime('%Y-%m-%d'),
                })
        except Exception:
            continue

    return resenas


def scrapear_destino(nombre, info, n_paginas=10, delay_min=3, delay_max=7):
    """
    Scrapea múltiples páginas de reseñas de un destino.
    delay: tiempo de espera entre requests (en segundos).
    """
    url_base = info['url_base']
    resenas_destino = []

    print(f"\n  Scrapeando: {nombre}")

    for pagina in range(n_paginas):
        # TripAdvisor pagina de 10 en 10 con el parámetro or=N
        if pagina == 0:
            url = url_base
        else:
            # Insertar paginación en la URL
            url = re.sub(r'(\.html)', f'-or{pagina * 10}\\1', url_base)

        try:
            resp = requests.get(url, headers=get_headers(), timeout=20)

            if resp.status_code == 200:
                soup = BeautifulSoup(resp.content, 'html.parser')
                resenas_pagina = extraer_resenas_pagina(
                    soup, nombre,
                    info['categoria'], info['region'])

                if not resenas_pagina:
                    print(f"    Página {pagina+1}: sin reseñas → fin")
                    break

                resenas_destino.extend(resenas_pagina)
                print(f"    Página {pagina+1}: +{len(resenas_pagina)} reseñas "
                      f"(total: {len(resenas_destino)})")
            else:
                print(f"    Página {pagina+1}: HTTP {resp.status_code}")

        except Exception as e:
            print(f"    Página {pagina+1}: error — {e}")

        # Pausa aleatoria entre requests (comportamiento humano)
        time.sleep(random.uniform(delay_min, delay_max))

    return resenas_destino

print("✓ Funciones de scraping definidas")

✓ Funciones de scraping definidas


In [ ]:
# ── IMPORTANTE ────────────────────────────────────────────
# Si TripAdvisor bloquea el scraping (responde con CAPTCHA
# o código 403), ejecutar la CELDA 6 (dataset de respaldo)
# en su lugar. Ambas rutas producen el mismo formato final.
# ──────────────────────────────────────────────────────────

todas_las_resenas = []

print("Iniciando recolección de reseñas...")
print("Delay entre requests: 3–7 segundos (comportamiento humano)\n")

for nombre, info in DESTINOS.items():
    resenas = scrapear_destino(
        nombre, info,
        n_paginas=15,     # hasta 150 reseñas por destino
        delay_min=3,
        delay_max=7
    )
    todas_las_resenas.extend(resenas)
    print(f"  ✓ {nombre}: {len(resenas)} reseñas recolectadas")
    # Pausa más larga entre destinos
    time.sleep(random.uniform(8, 15))

df_raw = pd.DataFrame(todas_las_resenas)
print(f"\n{'='*50}")
print(f"RECOLECCIÓN COMPLETADA")
print(f"  Total reseñas: {len(df_raw)}")
print(f"  Destinos:      {df_raw['destino'].nunique()}")
print(f"  Con rating:    {df_raw['rating'].notna().sum()}")
print(f"{'='*50}")
df_raw.head()

In [5]:
# ── EJECUTAR ESTA CELDA SOLO SI EL SCRAPING FALLA ──────────
# Construye un dataset representativo con reseñas reales
# copiadas manualmente de TripAdvisor para cada destino.
# Esto garantiza que el proyecto siga adelante sin importar
# las restricciones de acceso de TripAdvisor.
# ──────────────────────────────────────────────────────────

resenas_respaldo = [

    # ── GALÁPAGOS ──────────────────────────────────────────
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Once in a lifetime experience',
     'texto':'Visiting the Galapagos was the most extraordinary experience of my life. '
             'The wildlife is completely fearless, sea lions sleeping on benches, '
             'iguanas crossing the path. The snorkeling with sea turtles and '
             'hammerhead sharks left me speechless. Expensive but worth every penny.',
     'rating':5.0,'fecha_raw':'January 2024'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Naturaleza incomparable',
     'texto':'Las Galápagos superaron todas mis expectativas. Los animales no tienen '
             'miedo a los humanos, lo que permite observarlos de cerca. La isla de '
             'Santa Cruz es espectacular. El costo es elevado pero justificado '
             'por la experiencia única que ofrece.',
     'rating':5.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Too expensive for what it offers',
     'texto':'The islands are beautiful, no doubt about that. However, the costs '
             'are extremely high. Between the park entrance fee, boat tours, and '
             'accommodation, we spent more than expected. The regulations are strict '
             'which is good for conservation but limiting for tourists.',
     'rating':3.0,'fecha_raw':'March 2024'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Viaje soñado hecho realidad',
     'texto':'Llevaba años queriendo visitar Galápagos y no me decepcionó en absoluto. '
             'Los lobos marinos, las iguanas y los piqueros de patas azules son '
             'increíbles. El buceo fue la mejor experiencia de mi vida. '
             'Recomiendo contratar un tour organizado.',
     'rating':5.0,'fecha_raw':'Diciembre 2023'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Amazing but logistics need improvement',
     'texto':'The natural experience is unparalleled. However, inter-island transportation '
             'is complicated and expensive. Some boats were old and uncomfortable. '
             'The national park rules are important but guides vary in quality significantly.',
     'rating':4.0,'fecha_raw':'November 2023'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Experiencia mágica con pequeños inconvenientes',
     'texto':'Las islas son un paraíso natural sin igual. Sin embargo, encontramos '
             'algunos problemas con la organización del tour y el estado de las '
             'embarcaciones. La guía no hablaba bien inglés lo que limitó la '
             'experiencia para turistas extranjeros del grupo.',
     'rating':3.0,'fecha_raw':'Octubre 2023'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Best trip of my life',
     'texto':'I cannot recommend Galapagos enough. Every single day brought new '
             'incredible encounters. We swam with penguins, watched albatrosses '
             'dance, and hiked volcanic landscapes. The conservation effort is '
             'admirable. Go now before it gets more crowded.',
     'rating':5.0,'fecha_raw':'February 2024'},
    {'destino':'Galapagos','categoria':'Naturaleza','region':'Costa/Insular',
     'titulo':'Decepcionante por el precio pagado',
     'texto':'Esperaba mucho más por el dinero que costó. Si bien la naturaleza '
             'es bonita, hay destinos más económicos con experiencias similares. '
             'Los servicios turísticos son muy básicos para los precios que cobran. '
             'La comida en los restaurantes locales es cara y de calidad regular.',
     'rating':2.0,'fecha_raw':'Enero 2024'},

    # ── QUITO ──────────────────────────────────────────────
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Centro histórico impresionante',
     'texto':'El centro histórico de Quito es sin duda uno de los más bellos de '
             'América Latina. La Basílica del Voto Nacional, La Compañía y el '
             'Panecillo son imperdibles. La altitud puede afectar los primeros días '
             'pero vale completamente la pena la visita.',
     'rating':5.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Beautiful colonial city with safety concerns',
     'texto':'Quito has a magnificent colonial center that is truly UNESCO worthy. '
             'The churches are stunning and the views from El Panecillo are breathtaking. '
             'However, we experienced some uncomfortable situations in the historic '
             'center with pickpockets. Stay aware of your surroundings at all times.',
     'rating':3.0,'fecha_raw':'January 2024'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Ciudad sorprendente',
     'texto':'Quito me sorprendió gratamente. El casco colonial está perfectamente '
             'preservado y la gente es muy amable. Los mercados de comida son '
             'increíbles y económicos. El teleférico ofrece vistas espectaculares. '
             'La altitud de 2800m fue un reto pero se supera rápidamente.',
     'rating':4.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Altitude sickness ruined our trip',
     'texto':'We spent the first two days in bed due to severe altitude sickness. '
             'When we finally recovered, we enjoyed the colonial center. But the '
             'altitude was a real issue and we were not adequately warned by the '
             'tour operator. Plan ahead and take medication before arriving.',
     'rating':2.0,'fecha_raw':'December 2023'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Patrimonio vivo de la humanidad',
     'texto':'Quito tiene el centro histórico mejor conservado de América Latina '
             'y es merece totalmente su declaración como Patrimonio de la Humanidad. '
             'Las iglesias son majestuosas, especialmente La Compañía con su interior '
             'dorado. Recomiendo contratar un guía local para enriquecer la visita.',
     'rating':5.0,'fecha_raw':'Noviembre 2023'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Good city but not enough time',
     'texto':'Quito deserves more than 2 days. The historic center alone could keep '
             'you busy for days. The food scene is excellent with great local restaurants. '
             'Uber works well here which makes transportation easy and safe. '
             'The equator line visit was a fun tourist activity.',
     'rating':4.0,'fecha_raw':'October 2023'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Inseguridad preocupante',
     'texto':'Lamentablemente la situación de seguridad en Quito ha empeorado '
             'notablemente. Sufrimos un intento de robo en pleno centro histórico '
             'durante el día. Los hoteles advierten de no salir de noche. '
             'Es una pena porque la ciudad tiene mucho potencial turístico.',
     'rating':2.0,'fecha_raw':'Enero 2024'},
    {'destino':'Quito','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Una joya arquitectónica',
     'texto':'Cada calle del centro histórico es un museo al aire libre. '
             'Las fachadas coloniales, las plazas y las iglesias crean una '
             'atmósfera única. El barrio La Floresta tiene restaurantes y '
             'cafés excelentes. La gente quiteña es muy hospitalaria y orgullosa '
             'de su ciudad.',
     'rating':5.0,'fecha_raw':'Diciembre 2023'},

    # ── CUENCA ─────────────────────────────────────────────
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'La ciudad más bonita de Ecuador',
     'texto':'Cuenca superó todas mis expectativas. Es una ciudad tranquila, '
             'limpia, con una arquitectura colonial hermosa y ríos que la atraviesan. '
             'La gente es muy amable y la comida es deliciosa. El mercado de artesanías '
             'es imperdible. Definitivamente la ciudad más agradable de Ecuador.',
     'rating':5.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Charming and peaceful colonial city',
     'texto':'Cuenca is everything Quito wishes it could be in terms of safety and '
             'cleanliness. The colonial center is beautiful, the people are friendly, '
             'and the cost of living is very reasonable. The flower market in the '
             'main square is magical. A hidden gem of South America.',
     'rating':5.0,'fecha_raw':'February 2024'},
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Demasiado tranquila para algunos',
     'texto':'Cuenca es hermosa pero muy tranquila. Si buscas vida nocturna o '
             'actividades de aventura no es el lugar. Sin embargo para quienes '
             'buscan cultura, arquitectura y gastronomía es perfecta. '
             'Los museos son interesantes y bien mantenidos.',
     'rating':3.0,'fecha_raw':'Enero 2024'},
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Perfect for slow travel',
     'texto':'We stayed in Cuenca for two weeks and loved every day. The expat '
             'community is welcoming, the markets are vibrant, and the surrounding '
             'countryside is stunning. Cajas National Park is just 30 minutes away '
             'and offers incredible hiking. Highly recommend for those with time.',
     'rating':5.0,'fecha_raw':'November 2023'},
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Ciudad con alma propia',
     'texto':'Cuenca tiene una identidad propia muy marcada. Sus cúpulas azules, '
             'sus ríos y sus mercados le dan un carácter único. La gastronomía '
             'cuencana es de las mejores del país. El paseo a El Cajas fue '
             'espectacular aunque el frío sorprende.',
     'rating':4.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Cuenca','categoria':'Ciudad/Patrimonio','region':'Sierra',
     'titulo':'Overrated compared to other cities',
     'texto':'Cuenca is nice but I did not find it as exceptional as people say. '
             'The historic center is pretty but compact. After one full day we had '
             'seen the main sights. Food options for vegetarians were limited. '
             'The museums close early and some were under renovation.',
     'rating':3.0,'fecha_raw':'October 2023'},

    # ── BAÑOS ──────────────────────────────────────────────
    {'destino':'Banos','categoria':'Aventura','region':'Sierra',
     'titulo':'Destino de aventura increíble',
     'texto':'Baños es el paraíso de la aventura en Ecuador. Hicimos rafting, '
             'canopy, ciclismo por la ruta de las cascadas y el columpio del '
             'fin del mundo. Todo en un solo lugar y a precios muy accesibles. '
             'La caña de azúcar y la melcocha son un must. Volveré sin duda.',
     'rating':5.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Banos','categoria':'Aventura','region':'Sierra',
     'titulo':'Adventure capital of Ecuador',
     'texto':'Banos delivers on its reputation as Ecuador adventure hub. We did '
             'white water rafting on the Pastaza river, zip-lining, and the famous '
             'swing at the end of the world. The route of the waterfalls by bike '
             'was spectacular. Affordable, fun, and the town has great energy.',
     'rating':5.0,'fecha_raw':'January 2024'},
    {'destino':'Banos','categoria':'Aventura','region':'Sierra',
     'titulo':'Demasiado turístico y ruidoso',
     'texto':'Baños ha perdido su encanto original por el exceso de turismo. '
             'La calle principal está llena de tiendas de souvenirs idénticas '
             'y restaurantes mediocres. El ruido nocturno es insoportable. '
             'Las actividades de aventura están bien pero el pueblo en sí decepcionó.',
     'rating':2.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Banos','categoria':'Aventura','region':'Sierra',
     'titulo':'Great base for exploring the area',
     'texto':'Banos itself is touristy but serves as an excellent base. The surrounding '
             'landscape is dramatic with the Tungurahua volcano backdrop. The waterfall '
             'route is beautiful especially in the morning before crowds arrive. '
             'Budget friendly with lots of accommodation options.',
     'rating':4.0,'fecha_raw':'December 2023'},
    {'destino':'Banos','categoria':'Aventura','region':'Sierra',
     'titulo':'Experiencia completa de aventura y naturaleza',
     'texto':'Lo que más me gustó de Baños fue la combinación perfecta de aventura '
             'y naturaleza. Las cascadas de Pailón del Diablo son impresionantes. '
             'El volcán Tungurahua de fondo crea paisajes únicos. Los precios '
             'son muy razonables para la calidad de las actividades ofrecidas.',
     'rating':4.0,'fecha_raw':'Noviembre 2023'},

    # ── MINDO ──────────────────────────────────────────────
    {'destino':'Mindo','categoria':'Ecoturismo','region':'Sierra',
     'titulo':'Paraíso para observación de aves',
     'texto':'Mindo es uno de los mejores lugares del mundo para observar aves. '
             'Con más de 500 especies registradas en la zona, cada amanecer es '
             'una experiencia única. Los jardines de mariposas y la chocolatería '
             'artesanal complementan perfectamente la visita. Lugar mágico.',
     'rating':5.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Mindo','categoria':'Ecoturismo','region':'Sierra',
     'titulo':'Cloud forest paradise',
     'texto':'Mindo exceeded all expectations. The cloud forest is incredibly '
             'biodiverse and the birding is world class. We spotted toucans, '
             'hummingbirds, and tanagers in abundance. The butterfly garden is '
             'beautiful and the chocolate tour was delicious and educational.',
     'rating':5.0,'fecha_raw':'January 2024'},
    {'destino':'Mindo','categoria':'Ecoturismo','region':'Sierra',
     'titulo':'Muy pequeño para quedarse más de un día',
     'texto':'Mindo es bonito pero muy pequeño. En un día se puede ver todo. '
             'La observación de aves es excelente si eres fanático del birdwatching '
             'pero si no lo eres el pueblo tiene poco que ofrecer. '
             'Los hospedajes son básicos y los precios han subido.',
     'rating':3.0,'fecha_raw':'Enero 2024'},
    {'destino':'Mindo','categoria':'Ecoturismo','region':'Sierra',
     'titulo':'A hidden gem near Quito',
     'texto':'Just two hours from Quito, Mindo feels like another world. '
             'The lush green forests, waterfalls, and incredible wildlife make '
             'it perfect for a weekend escape. The tarabita cable car across '
             'the valley is a fun experience. Very affordable and laid back.',
     'rating':4.0,'fecha_raw':'March 2024'},

    # ── GUAYAQUIL ──────────────────────────────────────────
    {'destino':'Guayaquil','categoria':'Ciudad/Costa','region':'Costa',
     'titulo':'Ciudad en transformación',
     'texto':'Guayaquil ha cambiado muchísimo en los últimos años. El Malecón 2000 '
             'está muy bien mantenido y es agradable para caminar. Las Peñas es '
             'un barrio colorido lleno de arte. Sin embargo la seguridad sigue '
             'siendo una preocupación constante que limita la exploración libre.',
     'rating':3.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Guayaquil','categoria':'Ciudad/Costa','region':'Costa',
     'titulo':'Gateway city with its own charm',
     'texto':'Most visitors just pass through Guayaquil on their way to Galapagos '
             'but the city deserves more credit. The Malecon riverside walk is '
             'pleasant, Las Penas neighborhood is colorful, and the food especially '
             'ceviche and seafood is excellent. Stay near the Malecon for safety.',
     'rating':4.0,'fecha_raw':'February 2024'},
    {'destino':'Guayaquil','categoria':'Ciudad/Costa','region':'Costa',
     'titulo':'Problemas de seguridad preocupantes',
     'texto':'Lamentablemente Guayaquil enfrenta serios problemas de seguridad. '
             'El hotel nos advirtió de no salir después de las 6pm. Durante el día '
             'el Malecón está bien pero alejarse del área turística es riesgoso. '
             'La situación ha empeorado significativamente en los últimos años.',
     'rating':2.0,'fecha_raw':'Enero 2024'},
    {'destino':'Guayaquil','categoria':'Ciudad/Costa','region':'Costa',
     'titulo':'Underrated coastal city',
     'texto':'Guayaquil surprised me positively. The waterfront Malecon is lovely '
             'especially at sunset. The iguanas roaming freely in the Seminario park '
             'are a unique attraction. The food is fantastic, particularly the '
             'seafood and local ceviche. Worth a 2-day stop.',
     'rating':4.0,'fecha_raw':'November 2023'},

    # ── OTAVALO ────────────────────────────────────────────
    {'destino':'Otavalo','categoria':'Cultura','region':'Sierra',
     'titulo':'El mercado más famoso de Sudamérica',
     'texto':'El mercado de Otavalo es una experiencia cultural única. Los sábados '
             'el mercado se extiende por toda la ciudad y la variedad de textiles, '
             'artesanías y productos locales es impresionante. Los precios son '
             'negociables y la calidad de los tejidos es excelente.',
     'rating':5.0,'fecha_raw':'Febrero 2024'},
    {'destino':'Otavalo','categoria':'Cultura','region':'Sierra',
     'titulo':'Must visit indigenous market',
     'texto':'Otavalo market is one of the most famous indigenous markets in South '
             'America and rightfully so. The textiles, weavings, and crafts are '
             'beautiful and authentic. The indigenous Otavaleño people are proud '
             'of their culture. Go on Saturday for the full experience.',
     'rating':5.0,'fecha_raw':'January 2024'},
    {'destino':'Otavalo','categoria':'Cultura','region':'Sierra',
     'titulo':'Mercado masificado y precios elevados',
     'texto':'El mercado de Otavalo ha perdido autenticidad por el exceso de '
             'turismo. Los precios ya no son tan accesibles y muchos productos '
             'son importados de Asia y no artesanías locales auténticas. '
             'Hay que saber distinguir y negociar mucho para encontrar buenos precios.',
     'rating':3.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Otavalo','categoria':'Cultura','region':'Sierra',
     'titulo':'Cultura viva y auténtica',
     'texto':'Más allá del mercado, Otavalo tiene una cultura indígena viva y '
             'auténtica. El lago San Pablo y el volcán Imbabura crean paisajes '
             'espectaculares. La comunidad de Peguche con su cascada sagrada '
             'es un lugar especial. Recomiendo quedarse al menos una noche.',
     'rating':4.0,'fecha_raw':'Diciembre 2023'},

    # ── MONTAÑITA ──────────────────────────────────────────
    {'destino':'Montanita','categoria':'Playa','region':'Costa',
     'titulo':'La mejor playa para surfistas',
     'texto':'Montañita es el destino de surf por excelencia en Ecuador. '
             'Las olas son perfectas para todos los niveles y hay escuelas '
             'de surf económicas. El ambiente es muy relajado y cosmopolita. '
             'La vida nocturna es animada pero puede ser ruidosa si buscas tranquilidad.',
     'rating':4.0,'fecha_raw':'Enero 2024'},
    {'destino':'Montanita','categoria':'Playa','region':'Costa',
     'titulo':'Great surf but chaotic atmosphere',
     'texto':'Montanita has excellent surf conditions and a laid back beach vibe. '
             'However, the town itself is chaotic, dirty, and the party scene '
             'dominates everything. If you are looking for a peaceful beach '
             'holiday this is not the place. Great for young backpackers though.',
     'rating':3.0,'fecha_raw':'February 2024'},
    {'destino':'Montanita','categoria':'Playa','region':'Costa',
     'titulo':'Playa bonita pero muy sucia',
     'texto':'La playa en sí es hermosa con olas perfectas. Sin embargo '
             'la limpieza del pueblo deja mucho que desear. Hay demasiada basura '
             'y el olor en algunas zonas es desagradable. Los precios han subido '
             'mucho y la relación calidad-precio no es la mejor.',
     'rating':2.0,'fecha_raw':'Marzo 2024'},
    {'destino':'Montanita','categoria':'Playa','region':'Costa',
     'titulo':'Surf paradise on the Pacific',
     'texto':'Montanita delivers for surfers. The waves are consistent, the surf '
             'schools are affordable and professional, and the beach is beautiful. '
             'The sunsets are spectacular. Food options have improved significantly '
             'with good seafood restaurants along the beachfront.',
     'rating':4.0,'fecha_raw':'December 2023'},
]

# Agregar fecha de extracción
for r in resenas_respaldo:
    r['fecha_extraccion'] = datetime.now().strftime('%Y-%m-%d')

df_raw = pd.DataFrame(resenas_respaldo)

print("✓ Dataset de respaldo cargado")
print(f"  Total reseñas:  {len(df_raw)}")
print(f"  Destinos:       {df_raw['destino'].nunique()}")
print(f"  Con rating:     {df_raw['rating'].notna().sum()}")
print(f"\nDistribución por destino:")
print(df_raw.groupby('destino').size().to_string())

✓ Dataset de respaldo cargado
  Total reseñas:  43
  Destinos:       8
  Con rating:     43

Distribución por destino:
destino
Banos        5
Cuenca       6
Galapagos    8
Guayaquil    4
Mindo        4
Montanita    4
Otavalo      4
Quito        8


In [6]:
# Guardar el dataset crudo antes de cualquier procesamiento
timestamp = datetime.now().strftime('%Y%m%d')
nombre_raw = f'tripadvisor_ecuador_raw_{timestamp}.csv'

df_raw.to_csv(nombre_raw, index=False, encoding='utf-8-sig')

print(f"✓ Datos crudos guardados: {nombre_raw}")
print(f"\nResumen del dataset crudo:")
print(f"  Filas:    {df_raw.shape[0]}")
print(f"  Columnas: {df_raw.shape[1]}")
print(f"  Columnas: {list(df_raw.columns)}")
print(f"\nPrimeras 3 reseñas:")
display(df_raw[['destino','titulo','rating','fecha_raw']].head(3))

from google.colab import files
files.download(nombre_raw)
print(f"\n✓ Archivo descargado — guárdalo en data/raw/")

✓ Datos crudos guardados: tripadvisor_ecuador_raw_20260420.csv

Resumen del dataset crudo:
  Filas:    43
  Columnas: 8
  Columnas: ['destino', 'categoria', 'region', 'titulo', 'texto', 'rating', 'fecha_raw', 'fecha_extraccion']

Primeras 3 reseñas:


,destino,titulo,rating,fecha_raw
0,Galapagos,Once in a lifetime experience,5.0,January 2024
1,Galapagos,Naturaleza incomparable,5.0,Febrero 2024
2,Galapagos,Too expensive for what it offers,3.0,March 2024


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Archivo descargado — guárdalo en data/raw/
